# Final 01: DistilBERT Phase 1 Training, Calibration, and Mixed Emotion Prediction

This notebook is the final Phase 1 entry point. It trains or loads a DistilBERT classifier, calibrates confidence on a held-out calibration split, selects a routing threshold, evaluates the held-out test set, and exports Mixed Emotion Phase 1 predictions for Phase 2 reasoning.

Original exploratory notebooks are preserved. Use this final notebook for paper-ready experiments.

In [ ]:
# Colab setup. Run this first in a fresh GPU runtime.
# IMPORTANT: do not upgrade numpy/scipy inside an already-running Colab kernel.
# Colab already ships compatible numpy/scipy builds; upgrading them in-place can break scipy imports.
# After installation, restart the runtime once, then continue from the imports cell.
%pip install -q -U pandas scikit-learn matplotlib seaborn tqdm transformers datasets accelerate safetensors openpyxl wandb

import importlib.metadata as importlib_metadata
for package in ["numpy", "scipy", "pandas", "torch", "transformers", "datasets", "scikit-learn", "wandb"]:
    try:
        print(package, importlib_metadata.version(package))
    except Exception as exc:
        print(package, "not found", exc)

print("\nSETUP COMPLETE. Restart runtime once, then run from the imports cell.")
print("If scipy/numpy import errors still appear, use Runtime > Disconnect and delete runtime, then run this setup cell once again.")



## Imports

In [ ]:
import os
import gc
import json
import random
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import beta
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


## Configuration

The primary processed Reddit dataset is loaded from the existing GitHub media URL by default, so collaborators do not need to upload the input CSV manually. Google Drive is used for outputs only.

For the final paper run, keep `SAMPLES_PER_CLASS = 40000`. For a quick smoke test, set a smaller value such as `1000` or `3000`.

This notebook first runs an optional W&B sweep on the primary Reddit validation split, then trains the final DistilBERT model using the best hyperparameters. Mixed Emotion is loaded only after the final model, temperature, and routing threshold are fixed.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

# Primary curated Reddit dataset.
# This follows the existing large-file GitHub media input style used by the earlier DistilBERT Colab workflow.
PRIMARY_DATA_URL = (
    "https://media.githubusercontent.com/media/"
    "Branden-Kang/LLaMA-2/main/data/final_preprocessed_df2.csv"
)

# Optional local/Drive fallback. This is not required for collaborators if PRIMARY_DATA_URL is reachable.
PRIMARY_DATA_PATH = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/data/final_preprocessed_df.csv")
USE_PRIMARY_DRIVE_FALLBACK_FIRST = False

# Mixed Emotion v2.3 dataset is small and is loaded from this repository.
MIXED_EMOTION_DATA_URL = (
    "https://raw.githubusercontent.com/WoojinPark-Jay/"
    "confidence-guided-llm-reasoning-depression-risk-emotion/"
    "refs/heads/feature/phase2-mixed-emotion-reasoning-colab/data/supplementary/mixed_emotion/"
    "mixed_emotion_stress_test_v2_3_300.csv"
)
MIXED_EMOTION_LOCAL_PATH = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/data/mixed_emotion_stress_test_v2_3_300.csv")

TEXT_COLUMN = "title_with_selftext_cleaned"
LABEL_COLUMN = "class_group"
MIXED_TEXT_COLUMN = "text"
MIXED_LABEL_COLUMN = "target_label"

LABELS = ["Depression", "Neutral", "Happy"]
LABEL_TO_ID = {"Depression": 0, "Neutral": 1, "Happy": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

# Final run: 40000 per class. Smoke test: 1000 or 3000 per class.
SAMPLES_PER_CLASS = 40000
SAMPLING_MODE = "reservoir"  # "reservoir" reads full CSV; "first_balanced" is faster for smoke tests.
CSV_CHUNK_SIZE = 20000

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.10
CALIBRATION_RATIO = 0.10
TEST_RATIO = 0.10

MAX_LENGTH = 256
EARLY_STOPPING_PATIENCE = 2

# Fixed fallback hyperparameters. These are used if USE_WANDB_SWEEP=False.
DEFAULT_HYPERPARAMETERS = {
    "learning_rate": 3.8e-5,
    "batch_size": 32,
    "epochs": 3,
    "weight_decay": 0.01,
}

# W&B sweep. The final paper-oriented run should use this unless a compatible best run is intentionally reused.
USE_WANDB_SWEEP = True
WANDB_ENTITY = "kangsy413"
WANDB_PROJECT = "confidence-guided-distilbert-final"
WANDB_SWEEP_NAME = f"distilbert-final-{SAMPLES_PER_CLASS}-per-class-validation-f1"
WANDB_SWEEP_COUNT = 4
WANDB_OBJECTIVE_METRIC = "validation_f1_macro"
WANDB_OBJECTIVE_GOAL = "maximize"
LOG_FINAL_TRAINING_TO_WANDB = True

WANDB_SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {"name": WANDB_OBJECTIVE_METRIC, "goal": WANDB_OBJECTIVE_GOAL},
    "parameters": {
        "learning_rate": {"min": 1e-5, "max": 2e-4},
        "batch_size": {"values": [16, 32, 64]},
        "epochs": {"values": [3, 5]},
        "weight_decay": {"values": [1e-2, 1e-3, 1e-4]},
    },
}

# Risk/coverage threshold calibration.
TARGET_SELECTIVE_RISK = 0.05
RISK_CONFIDENCE_DELTA = 0.05
MIN_ACCEPTED_FRACTION = 0.10
THRESHOLD_GRID = np.round(np.arange(0.50, 1.001, 0.01), 2)

RUN_TRAINING = True
LOAD_EXISTING_MODEL_IF_AVAILABLE = True


In [ ]:
# Persistent output configuration.
# The notebook stops if Google Drive is unavailable. This prevents long runs from saving only to /content.
USE_GOOGLE_DRIVE_OUTPUT = True
REQUIRE_PERSISTENT_OUTPUT = True
LOCAL_OUTPUT_ROOT = Path("outputs_final")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final")

OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
DRIVE_OUTPUT_AVAILABLE = False

if USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        probe_path = DRIVE_OUTPUT_ROOT / "_drive_write_test.txt"
        probe_path.write_text("ok", encoding="utf-8")
        probe_path.unlink(missing_ok=True)
        OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
        DRIVE_OUTPUT_AVAILABLE = True
        print(f"Google Drive output enabled: {OUTPUT_ROOT}")
    except Exception as exc:
        if REQUIRE_PERSISTENT_OUTPUT:
            raise RuntimeError(
                "Google Drive output is unavailable, so the notebook stopped before running expensive work. "
                "Fix Drive authorization/mount first, or set REQUIRE_PERSISTENT_OUTPUT = False only for a temporary smoke test."
            ) from exc
        print(f"Google Drive output is unavailable ({exc}); falling back to local runtime output.")
        OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
else:
    if REQUIRE_PERSISTENT_OUTPUT:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE_OUTPUT is False while REQUIRE_PERSISTENT_OUTPUT is True. "
            "Turn on Drive output or set REQUIRE_PERSISTENT_OUTPUT = False for a temporary run."
        )

PHASE1_OUTPUT_DIR = OUTPUT_ROOT / "phase1_distilbert"
PHASE1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Phase 1 output directory:", PHASE1_OUTPUT_DIR)


## Optional W&B Login

In [ ]:
WANDB_AVAILABLE = False
if USE_WANDB_SWEEP or LOG_FINAL_TRAINING_TO_WANDB:
    try:
        from google.colab import userdata
        import wandb
        key = userdata.get("WANDB_API_KEY")
        if key:
            wandb.login(key=key)
        else:
            wandb.login()
        os.environ["WANDB_PROJECT"] = WANDB_PROJECT
        WANDB_AVAILABLE = True
        print("W&B enabled:", WANDB_PROJECT)
    except Exception as exc:
        raise RuntimeError("W&B is required for this configuration. Add WANDB_API_KEY in Colab secrets or set USE_WANDB_SWEEP=False and LOG_FINAL_TRAINING_TO_WANDB=False.") from exc
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled for this run.")


## Load and Sample Primary Dataset

In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, np.integer)) and int(value) in ID_TO_LABEL:
        return int(value)
    if isinstance(value, (float, np.floating)) and float(value).is_integer() and int(value) in ID_TO_LABEL:
        return int(value)
    text = str(value).strip().lower().replace("-", "_").replace(" ", "_")
    if text in {"0", "depression", "depressed", "depression_group"} or "depress" in text:
        return 0
    if text in {"1", "neutral", "neutral_group"} or "neutral" in text:
        return 1
    if text in {"2", "happy", "happiness", "happy_group", "positive"} or "happy" in text:
        return 2
    return None


def primary_source():
    if USE_PRIMARY_DRIVE_FALLBACK_FIRST and PRIMARY_DATA_PATH.exists():
        return str(PRIMARY_DATA_PATH)
    if PRIMARY_DATA_URL:
        return PRIMARY_DATA_URL
    if PRIMARY_DATA_PATH.exists():
        return str(PRIMARY_DATA_PATH)
    raise FileNotFoundError(
        "No primary dataset source is available. Check PRIMARY_DATA_URL or PRIMARY_DATA_PATH."
    )


def sample_balanced_from_csv(source, text_col, label_col, samples_per_class, chunksize=20000, mode="reservoir", seed=42):
    if mode not in {"reservoir", "first_balanced"}:
        raise ValueError("SAMPLING_MODE must be 'reservoir' or 'first_balanced'.")
    rng = random.Random(seed)
    reservoirs = {0: [], 1: [], 2: []}
    seen = {0: 0, 1: 0, 2: 0}
    usecols = [text_col, label_col]
    for chunk_idx, chunk in enumerate(pd.read_csv(source, usecols=usecols, chunksize=chunksize, low_memory=False), start=1):
        chunk = chunk.dropna(subset=usecols).copy()
        chunk["label"] = chunk[label_col].map(normalize_label)
        chunk = chunk[chunk["label"].isin([0, 1, 2])].copy()
        chunk["text"] = chunk[text_col].astype(str).str.strip()
        chunk = chunk[chunk["text"].str.len() > 0]
        if mode == "first_balanced":
            chunk = chunk.sample(frac=1.0, random_state=seed + chunk_idx)
            for class_id in [0, 1, 2]:
                remain = samples_per_class - len(reservoirs[class_id])
                if remain > 0:
                    reservoirs[class_id].extend(chunk[chunk["label"] == class_id][["text", "label"]].head(remain).to_dict("records"))
            if all(len(v) >= samples_per_class for v in reservoirs.values()):
                print(f"Early stop after chunk {chunk_idx}.")
                break
        else:
            for row in chunk[["text", "label"]].to_dict("records"):
                class_id = int(row["label"])
                seen[class_id] += 1
                if len(reservoirs[class_id]) < samples_per_class:
                    reservoirs[class_id].append(row)
                else:
                    j = rng.randint(0, seen[class_id] - 1)
                    if j < samples_per_class:
                        reservoirs[class_id][j] = row
        if chunk_idx % 10 == 0:
            print("processed chunks", chunk_idx, {ID_TO_LABEL[k]: len(v) for k, v in reservoirs.items()})
    for class_id, rows in reservoirs.items():
        if len(rows) < samples_per_class:
            raise ValueError(f"Only {len(rows)} rows found for {ID_TO_LABEL[class_id]}, required {samples_per_class}.")
    out = pd.DataFrame([r for class_id in [0,1,2] for r in reservoirs[class_id][:samples_per_class]])
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    out["label_str"] = out["label"].map(ID_TO_LABEL)
    out.insert(0, "example_id", [f"RED_{i:06d}" for i in range(len(out))])
    return out

RUN_SEED = globals().get("SEED", 42)
source = primary_source()
print("Primary source:", source)
primary_df = sample_balanced_from_csv(source, TEXT_COLUMN, LABEL_COLUMN, SAMPLES_PER_CLASS, CSV_CHUNK_SIZE, SAMPLING_MODE, RUN_SEED)
print(primary_df.shape)
display(primary_df.head())
print(primary_df["label_str"].value_counts())
primary_df.to_csv(PHASE1_OUTPUT_DIR / "phase1_training_sample.csv", index=False)



## Split Train / Validation / Calibration / Test

In [ ]:
train_df, temp_df = train_test_split(
    primary_df,
    test_size=(1.0 - TRAIN_RATIO),
    stratify=primary_df["label"],
    random_state=SEED,
)
relative_val = VALIDATION_RATIO / (VALIDATION_RATIO + CALIBRATION_RATIO + TEST_RATIO)
val_df, cal_test_df = train_test_split(
    temp_df,
    test_size=(1.0 - relative_val),
    stratify=temp_df["label"],
    random_state=SEED,
)
relative_cal = CALIBRATION_RATIO / (CALIBRATION_RATIO + TEST_RATIO)
calibration_df, test_df = train_test_split(
    cal_test_df,
    test_size=(1.0 - relative_cal),
    stratify=cal_test_df["label"],
    random_state=SEED,
)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "calibration", "test"],
    "rows": [len(train_df), len(val_df), len(calibration_df), len(test_df)],
})
display(split_summary)
for name, part in [("train", train_df), ("validation", val_df), ("calibration", calibration_df), ("test", test_df)]:
    print("\n", name)
    print(part["label_str"].value_counts().sort_index())
    part.to_csv(PHASE1_OUTPUT_DIR / f"phase1_{name}_split.csv", index=False)
split_summary.to_csv(PHASE1_OUTPUT_DIR / "phase1_split_summary.csv", index=False)


## Tokenization and Training

In [ ]:
import inspect
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    return Dataset.from_pandas(df[["example_id", "text", "label"]].reset_index(drop=True), preserve_index=False)

datasets = DatasetDict({
    "train": to_dataset(train_df),
    "validation": to_dataset(val_df),
    "calibration": to_dataset(calibration_df),
    "test": to_dataset(test_df),
})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = datasets.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall,
    }

FINAL_MODEL_DIR = PHASE1_OUTPUT_DIR / "distilbert_best_model"
SWEEP_RESULTS_PATH = PHASE1_OUTPUT_DIR / "wandb_sweep_results.jsonl"
BEST_HYPERPARAMETERS_PATH = PHASE1_OUTPUT_DIR / "best_hyperparameters.json"


def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
    )


def training_args_for(run_dir, params, run_name, report_to):
    return TrainingArguments(
        output_dir=str(run_dir),
        learning_rate=float(params["learning_rate"]),
        per_device_train_batch_size=int(params["batch_size"]),
        per_device_eval_batch_size=int(params["batch_size"]),
        num_train_epochs=int(params["epochs"]),
        weight_decay=float(params["weight_decay"]),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=100,
        report_to=report_to,
        run_name=run_name,
        seed=SEED,
    )


def trainer_init_kwargs(**kwargs):
    signature = inspect.signature(Trainer.__init__)
    if "processing_class" in signature.parameters:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in signature.parameters:
        kwargs["tokenizer"] = tokenizer
    return kwargs


def build_trainer(**kwargs):
    return Trainer(**trainer_init_kwargs(**kwargs))


def run_one_training(params, run_dir, run_name, report_to="none", save_model=False):
    model = build_model()
    args = training_args_for(run_dir, params, run_name, report_to)
    trainer_local = build_trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    trainer_local.train()
    eval_metrics = trainer_local.evaluate(tokenized["validation"], metric_key_prefix="validation")
    if save_model:
        trainer_local.model.save_pretrained(FINAL_MODEL_DIR)
        tokenizer.save_pretrained(FINAL_MODEL_DIR)
    return trainer_local, eval_metrics


def run_wandb_sweep():
    import wandb
    sweep_results = []

    def sweep_objective():
        with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY) as run:
            params = dict(wandb.config)
            run_dir = PHASE1_OUTPUT_DIR / "wandb_sweep_runs" / run.id
            trainer_local, metrics = run_one_training(
                params=params,
                run_dir=run_dir,
                run_name=f"sweep-{run.id}",
                report_to="wandb",
                save_model=False,
            )
            result = {
                "run_id": run.id,
                "run_name": run.name,
                "params": params,
                "validation_accuracy": float(metrics.get("validation_accuracy", np.nan)),
                "validation_f1_macro": float(metrics.get("validation_f1_macro", np.nan)),
                "validation_precision_macro": float(metrics.get("validation_precision_macro", np.nan)),
                "validation_recall_macro": float(metrics.get("validation_recall_macro", np.nan)),
            }
            wandb.log(result)
            sweep_results.append(result)
            with open(SWEEP_RESULTS_PATH, "a", encoding="utf-8") as f:
                f.write(json.dumps(result) + "\n")
            del trainer_local
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    sweep_id = wandb.sweep(
        sweep=WANDB_SWEEP_CONFIG,
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
    )
    print("Created W&B sweep:", sweep_id)
    wandb.agent(sweep_id, function=sweep_objective, count=WANDB_SWEEP_COUNT, project=WANDB_PROJECT, entity=WANDB_ENTITY)

    if not sweep_results:
        raise RuntimeError("W&B sweep completed without local results.")
    results_df = pd.DataFrame(sweep_results).sort_values("validation_f1_macro", ascending=False)
    results_df.to_csv(PHASE1_OUTPUT_DIR / "wandb_sweep_results.csv", index=False)
    best = results_df.iloc[0].to_dict()
    return best["params"], results_df


if LOAD_EXISTING_MODEL_IF_AVAILABLE and FINAL_MODEL_DIR.exists() and BEST_HYPERPARAMETERS_PATH.exists():
    print("Loading existing final model and hyperparameters:", FINAL_MODEL_DIR)
    BEST_HYPERPARAMETERS = json.loads(BEST_HYPERPARAMETERS_PATH.read_text(encoding="utf-8"))
    model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_MODEL_DIR), num_labels=3)
else:
    if not RUN_TRAINING:
        raise RuntimeError("RUN_TRAINING=False but no existing final model was found.")

    if USE_WANDB_SWEEP:
        BEST_HYPERPARAMETERS, sweep_df = run_wandb_sweep()
        display(sweep_df.head())
    else:
        BEST_HYPERPARAMETERS = DEFAULT_HYPERPARAMETERS.copy()

    BEST_HYPERPARAMETERS = {
        "learning_rate": float(BEST_HYPERPARAMETERS["learning_rate"]),
        "batch_size": int(BEST_HYPERPARAMETERS["batch_size"]),
        "epochs": int(BEST_HYPERPARAMETERS["epochs"]),
        "weight_decay": float(BEST_HYPERPARAMETERS["weight_decay"]),
    }
    BEST_HYPERPARAMETERS_PATH.write_text(json.dumps(BEST_HYPERPARAMETERS, indent=2), encoding="utf-8")
    print("Best hyperparameters:", BEST_HYPERPARAMETERS)

    final_report_to = "wandb" if LOG_FINAL_TRAINING_TO_WANDB else "none"
    final_trainer, final_validation_metrics = run_one_training(
        params=BEST_HYPERPARAMETERS,
        run_dir=PHASE1_OUTPUT_DIR / "final_training_checkpoints",
        run_name=f"final-distilbert-{SAMPLES_PER_CLASS}-per-class",
        report_to=final_report_to,
        save_model=True,
    )
    model = final_trainer.model
    pd.DataFrame([final_validation_metrics]).to_csv(PHASE1_OUTPUT_DIR / "final_validation_metrics.csv", index=False)
    print("Saved final best model:", FINAL_MODEL_DIR)

trainer = build_trainer(model=model, data_collator=data_collator, compute_metrics=compute_metrics)
print("Final model ready. Best hyperparameters:", BEST_HYPERPARAMETERS)



## Prediction Helpers

In [ ]:
def predict_logits(df, batch_size=BATCH_SIZE):
    ds = to_dataset(df).map(tokenize, batched=True)
    pred = trainer.predict(ds)
    logits = pred.predictions
    labels = pred.label_ids
    return logits, labels


def prediction_frame(df, logits, temperature=1.0):
    raw_probs = softmax(logits, axis=1)
    cal_probs = softmax(logits / temperature, axis=1)
    raw_pred = raw_probs.argmax(axis=1)
    cal_pred = cal_probs.argmax(axis=1)
    out = df[["example_id", "text", "label", "label_str"]].copy().reset_index(drop=True)
    out["phase1_label"] = [ID_TO_LABEL[int(i)] for i in cal_pred]
    out["phase1_confidence"] = cal_probs.max(axis=1)
    out["phase1_raw_label"] = [ID_TO_LABEL[int(i)] for i in raw_pred]
    out["phase1_raw_confidence"] = raw_probs.max(axis=1)
    for class_id, label in ID_TO_LABEL.items():
        out[f"phase1_probability_{label.lower()}"] = cal_probs[:, class_id]
        out[f"phase1_raw_probability_{label.lower()}"] = raw_probs[:, class_id]
    return out


def nll_for_temperature(temp, logits, labels):
    probs = softmax(logits / temp, axis=1)
    eps = 1e-12
    return -np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)))


def fit_temperature(logits, labels):
    result = minimize_scalar(lambda t: nll_for_temperature(t, logits, labels), bounds=(0.05, 10.0), method="bounded")
    return float(result.x), float(result.fun)


def clopper_pearson_upper(errors, n, alpha=0.05):
    if n <= 0:
        return np.nan
    if errors >= n:
        return 1.0
    return float(beta.ppf(1 - alpha, errors + 1, n - errors))


def threshold_table(pred_df, threshold_grid=THRESHOLD_GRID):
    rows = []
    y_true = pred_df["label_str"].values
    y_pred = pred_df["phase1_label"].values
    conf = pred_df["phase1_confidence"].values
    for tau in threshold_grid:
        accepted = conf >= tau
        n = int(accepted.sum())
        errors = int((y_true[accepted] != y_pred[accepted]).sum()) if n else 0
        coverage = n / len(pred_df)
        risk = errors / n if n else np.nan
        rows.append({
            "threshold": tau,
            "accepted_count": n,
            "routed_count": int(len(pred_df) - n),
            "coverage": coverage,
            "routing_rate": 1.0 - coverage,
            "selective_risk": risk,
            "selective_risk_upper_bound": clopper_pearson_upper(errors, n, RISK_CONFIDENCE_DELTA),
        })
    return pd.DataFrame(rows)


def select_threshold(table):
    min_accepted = int(np.ceil(MIN_ACCEPTED_FRACTION * len(calibration_df)))
    candidates = table[(table["accepted_count"] >= min_accepted) & (table["selective_risk_upper_bound"] <= TARGET_SELECTIVE_RISK)].copy()
    if candidates.empty:
        print("No threshold satisfied the target upper-bound risk. Falling back to lowest empirical risk then highest coverage.")
        candidates = table.sort_values(["selective_risk", "coverage"], ascending=[True, False]).head(1)
    else:
        candidates = candidates.sort_values(["coverage", "threshold"], ascending=[False, True]).head(1)
    return candidates.iloc[0]


## Calibration, Threshold Selection, and Test Evaluation

In [ ]:
cal_logits, cal_labels = predict_logits(calibration_df)
temperature, calibration_nll = fit_temperature(cal_logits, cal_labels)
print("Optimal temperature:", temperature)
print("Calibration NLL:", calibration_nll)

cal_pred_df = prediction_frame(calibration_df, cal_logits, temperature)
th_table = threshold_table(cal_pred_df)
selected = select_threshold(th_table)
ROUTING_THRESHOLD = float(selected["threshold"])
print("Selected routing threshold:", ROUTING_THRESHOLD)
display(selected.to_frame().T)

th_table.to_csv(PHASE1_OUTPUT_DIR / "phase1_threshold_calibration_table.csv", index=False)
selected.to_frame().T.to_csv(PHASE1_OUTPUT_DIR / "phase1_selected_threshold.csv", index=False)

# Held-out test evaluation with fixed temperature and fixed threshold.
test_logits, test_labels = predict_logits(test_df)
test_pred_df = prediction_frame(test_df, test_logits, temperature)
test_pred_df["phase1_accepted"] = test_pred_df["phase1_confidence"] >= ROUTING_THRESHOLD
test_pred_df["phase1_routed"] = ~test_pred_df["phase1_accepted"]
test_pred_df["routing_threshold"] = ROUTING_THRESHOLD
test_pred_df["temperature"] = temperature

test_accuracy = accuracy_score(test_pred_df["label_str"], test_pred_df["phase1_label"])
test_macro = precision_recall_fscore_support(test_pred_df["label_str"], test_pred_df["phase1_label"], average="macro", zero_division=0)
print("Test accuracy:", test_accuracy)
print(classification_report(test_pred_df["label_str"], test_pred_df["phase1_label"], labels=LABELS, zero_division=0))

test_pred_df.to_csv(PHASE1_OUTPUT_DIR / "phase1_test_predictions.csv", index=False)
summary = {
    "samples_per_class": SAMPLES_PER_CLASS,
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "calibration_rows": len(calibration_df),
    "test_rows": len(test_df),
    "temperature": temperature,
    "routing_threshold": ROUTING_THRESHOLD,
    "test_accuracy": float(test_accuracy),
    "test_precision_macro": float(test_macro[0]),
    "test_recall_macro": float(test_macro[1]),
    "test_f1_macro": float(test_macro[2]),
    "test_coverage": float(test_pred_df["phase1_accepted"].mean()),
    "test_routing_rate": float(test_pred_df["phase1_routed"].mean()),
}
(PHASE1_OUTPUT_DIR / "distilbert_phase1_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
pd.DataFrame([summary]).to_csv(PHASE1_OUTPUT_DIR / "distilbert_phase1_summary.csv", index=False)
display(pd.DataFrame([summary]))


## Advanced Confidence-Threshold Analysis

This section exports the paper-defense outputs for the confidence-guided routing method: calibration metrics, reliability diagrams, risk-coverage curves, threshold sensitivity, score ablations, bootstrap confidence intervals, threshold stability, per-class selective risk, high-confidence errors, and reproducibility metadata. It uses the already trained final DistilBERT model and the fixed temperature/threshold selected above.


In [ ]:
ADVANCED_THRESHOLD_DIR = PHASE1_OUTPUT_DIR / "advanced_confidence_threshold_analysis"
ADVANCED_THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_ECE_BINS = 15
THRESHOLD_BOOTSTRAPS = 200
BOOTSTRAP_RANDOM_STATE = SEED
TOP_HIGH_CONFIDENCE_ERRORS = 100


def _probability_matrix(df, prefix="phase1_probability"):
    return df[[f"{prefix}_{label.lower()}" for label in LABELS]].to_numpy(dtype=float)


def _prediction_ids(df, pred_col="phase1_label"):
    return df[pred_col].map(LABEL_TO_ID).to_numpy(dtype=int)


def _true_ids(df):
    return df["label_str"].map(LABEL_TO_ID).to_numpy(dtype=int)


def multiclass_brier_score(probs, labels):
    y = np.eye(len(LABELS))[labels]
    return float(np.mean(np.sum((probs - y) ** 2, axis=1)))


def negative_log_likelihood(probs, labels):
    eps = 1e-12
    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], eps, 1.0))))


def ece_table(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        count = int(mask.sum())
        if count:
            acc = float(correct[mask].mean())
            conf = float(confidences[mask].mean())
            gap = abs(acc - conf)
            ece += (count / len(labels)) * gap
        else:
            acc = conf = gap = np.nan
        rows.append({
            "bin": i + 1,
            "lower": lo,
            "upper": hi,
            "count": count,
            "accuracy": acc,
            "mean_confidence": conf,
            "abs_gap": gap,
        })
    return pd.DataFrame(rows), float(ece)


def adaptive_ece_table(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(float)
    order = np.argsort(confidences)
    splits = np.array_split(order, n_bins)
    rows = []
    ece = 0.0
    for i, idx in enumerate(splits):
        if len(idx) == 0:
            continue
        acc = float(correct[idx].mean())
        conf = float(confidences[idx].mean())
        gap = abs(acc - conf)
        ece += (len(idx) / len(labels)) * gap
        rows.append({
            "bin": i + 1,
            "lower": float(confidences[idx].min()),
            "upper": float(confidences[idx].max()),
            "count": int(len(idx)),
            "accuracy": acc,
            "mean_confidence": conf,
            "abs_gap": gap,
        })
    return pd.DataFrame(rows), float(ece)


def confidence_scores(df):
    probs = _probability_matrix(df)
    sorted_probs = np.sort(probs, axis=1)
    entropy = -np.sum(np.clip(probs, 1e-12, 1.0) * np.log(np.clip(probs, 1e-12, 1.0)), axis=1)
    out = df.copy()
    out["confidence_msp"] = probs.max(axis=1)
    out["confidence_margin"] = sorted_probs[:, -1] - sorted_probs[:, -2]
    out["confidence_negative_entropy"] = -entropy
    return out


def selective_metrics_at_threshold(df, score_col, threshold):
    accepted = df[score_col].to_numpy(dtype=float) >= threshold
    n = int(accepted.sum())
    total = len(df)
    errors = int((df.loc[accepted, "label_str"] != df.loc[accepted, "phase1_label"]).sum()) if n else 0
    coverage = n / total if total else np.nan
    risk = errors / n if n else np.nan
    return {
        "threshold": float(threshold),
        "accepted_count": n,
        "routed_count": int(total - n),
        "coverage": float(coverage),
        "routing_rate": float(1.0 - coverage),
        "selective_risk": float(risk) if not pd.isna(risk) else np.nan,
        "selective_risk_upper_bound": clopper_pearson_upper(errors, n, RISK_CONFIDENCE_DELTA),
        "accepted_errors": errors,
    }


def score_threshold_sweep(df, score_col, thresholds=THRESHOLD_GRID):
    rows = []
    for tau in thresholds:
        row = selective_metrics_at_threshold(df, score_col, tau)
        row["score"] = score_col
        rows.append(row)
    return pd.DataFrame(rows)


def select_from_sweep(sweep, total_n=None):
    reference_n = len(cal_pred_df) if total_n is None else int(total_n)
    min_accepted = int(np.ceil(MIN_ACCEPTED_FRACTION * reference_n))
    candidates = sweep[(sweep["accepted_count"] >= min_accepted) & (sweep["selective_risk_upper_bound"] <= TARGET_SELECTIVE_RISK)].copy()
    if candidates.empty:
        candidates = sweep.sort_values(["selective_risk", "coverage"], ascending=[True, False]).head(1)
        reason = "fallback_lowest_empirical_risk"
    else:
        candidates = candidates.sort_values(["coverage", "threshold"], ascending=[False, True]).head(1)
        reason = "upper_bound_risk_control"
    selected_row = candidates.iloc[0].copy()
    selected_row["selection_reason"] = reason
    return selected_row


def risk_coverage_auc(df):
    ordered = df.sort_values("phase1_confidence", ascending=False).reset_index(drop=True)
    errors = (ordered["label_str"] != ordered["phase1_label"]).astype(int).to_numpy()
    cumulative_errors = np.cumsum(errors)
    k = np.arange(1, len(ordered) + 1)
    coverage = k / len(ordered)
    risk = cumulative_errors / k
    aurc = float(np.trapz(risk, coverage))
    return pd.DataFrame({"coverage": coverage, "selective_risk": risk, "accepted_count": k}), aurc


def bootstrap_metrics(df, threshold, n_bootstraps=200):
    rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
    rows = []
    for b in range(n_bootstraps):
        sample_idx = rng.integers(0, len(df), len(df))
        sample = df.iloc[sample_idx].reset_index(drop=True)
        row = selective_metrics_at_threshold(sample, "phase1_confidence", threshold)
        row["bootstrap_id"] = b
        rows.append(row)
    boot = pd.DataFrame(rows)
    summary_rows = []
    for metric in ["coverage", "routing_rate", "selective_risk", "selective_risk_upper_bound"]:
        values = boot[metric].dropna().to_numpy()
        summary_rows.append({
            "metric": metric,
            "mean": float(np.mean(values)) if len(values) else np.nan,
            "ci_lower_2_5": float(np.percentile(values, 2.5)) if len(values) else np.nan,
            "ci_upper_97_5": float(np.percentile(values, 97.5)) if len(values) else np.nan,
        })
    return boot, pd.DataFrame(summary_rows)


def threshold_stability_bootstrap(df, n_bootstraps=200):
    rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE + 1)
    rows = []
    for b in range(n_bootstraps):
        sample_idx = rng.integers(0, len(df), len(df))
        sample = df.iloc[sample_idx].reset_index(drop=True)
        sweep = score_threshold_sweep(sample, "phase1_confidence", THRESHOLD_GRID)
        selected_row = select_from_sweep(sweep)
        rows.append({
            "bootstrap_id": b,
            "selected_threshold": float(selected_row["threshold"]),
            "coverage": float(selected_row["coverage"]),
            "selective_risk": float(selected_row["selective_risk"]),
            "selective_risk_upper_bound": float(selected_row["selective_risk_upper_bound"]),
            "selection_reason": selected_row["selection_reason"],
        })
    return pd.DataFrame(rows)


def per_class_selective_metrics(df, threshold):
    rows = []
    for label in LABELS:
        part = df[df["label_str"] == label].copy()
        if part.empty:
            continue
        row = selective_metrics_at_threshold(part, "phase1_confidence", threshold)
        row["class_label"] = label
        rows.append(row)
    return pd.DataFrame(rows)


def class_conditional_thresholds(df):
    rows = []
    for label in LABELS:
        part = df[df["label_str"] == label].copy()
        if len(part) < 5:
            rows.append({"class_label": label, "selected_threshold": ROUTING_THRESHOLD, "selection_reason": "global_fallback_small_class"})
            continue
        sweep = score_threshold_sweep(part, "phase1_confidence", THRESHOLD_GRID)
        selected_row = select_from_sweep(sweep, total_n=len(part))
        row = selected_row.to_dict()
        row["class_label"] = label
        row["selected_threshold"] = row.pop("threshold")
        rows.append(row)
        sweep.to_csv(ADVANCED_THRESHOLD_DIR / f"class_{label.lower()}_threshold_sweep.csv", index=False)
    return pd.DataFrame(rows)


def input_length_analysis(df, threshold):
    temp = df.copy()
    temp["text_length_chars"] = temp["text"].astype(str).str.len()
    temp["text_length_tokens_proxy"] = temp["text"].astype(str).str.split().str.len()
    temp["length_bin"] = pd.qcut(temp["text_length_tokens_proxy"].rank(method="first"), q=4, labels=["Q1_short", "Q2", "Q3", "Q4_long"])
    rows = []
    for bin_name, part in temp.groupby("length_bin", observed=False):
        row = selective_metrics_at_threshold(part, "phase1_confidence", threshold)
        row["length_bin"] = str(bin_name)
        row["mean_tokens_proxy"] = float(part["text_length_tokens_proxy"].mean())
        rows.append(row)
    return pd.DataFrame(rows)


cal_adv = confidence_scores(cal_pred_df)
test_adv = confidence_scores(test_pred_df)

cal_probs_raw = _probability_matrix(cal_adv, prefix="phase1_raw_probability")
cal_probs_scaled = _probability_matrix(cal_adv, prefix="phase1_probability")
test_probs_scaled = _probability_matrix(test_adv, prefix="phase1_probability")
cal_y = _true_ids(cal_adv)
test_y = _true_ids(test_adv)

raw_ece_table, raw_ece = ece_table(cal_probs_raw, cal_y, PRIMARY_ECE_BINS)
scaled_ece_table, scaled_ece = ece_table(cal_probs_scaled, cal_y, PRIMARY_ECE_BINS)
adaptive_table, adaptive_ece = adaptive_ece_table(cal_probs_scaled, cal_y, PRIMARY_ECE_BINS)

raw_ece_table.to_csv(ADVANCED_THRESHOLD_DIR / "calibration_reliability_raw_msp.csv", index=False)
scaled_ece_table.to_csv(ADVANCED_THRESHOLD_DIR / "calibration_reliability_temperature_scaled_msp.csv", index=False)
adaptive_table.to_csv(ADVANCED_THRESHOLD_DIR / "adaptive_calibration_reliability_temperature_scaled_msp.csv", index=False)

calibration_metric_summary = pd.DataFrame([
    {
        "split": "calibration",
        "confidence_method": "raw_msp",
        "ece": raw_ece,
        "adaptive_ece": np.nan,
        "brier_score": multiclass_brier_score(cal_probs_raw, cal_y),
        "nll": negative_log_likelihood(cal_probs_raw, cal_y),
        "temperature": 1.0,
    },
    {
        "split": "calibration",
        "confidence_method": "temperature_scaled_msp",
        "ece": scaled_ece,
        "adaptive_ece": adaptive_ece,
        "brier_score": multiclass_brier_score(cal_probs_scaled, cal_y),
        "nll": negative_log_likelihood(cal_probs_scaled, cal_y),
        "temperature": temperature,
    },
    {
        "split": "test",
        "confidence_method": "temperature_scaled_msp",
        "ece": ece_table(test_probs_scaled, test_y, PRIMARY_ECE_BINS)[1],
        "adaptive_ece": adaptive_ece_table(test_probs_scaled, test_y, PRIMARY_ECE_BINS)[1],
        "brier_score": multiclass_brier_score(test_probs_scaled, test_y),
        "nll": negative_log_likelihood(test_probs_scaled, test_y),
        "temperature": temperature,
    },
])
calibration_metric_summary.to_csv(ADVANCED_THRESHOLD_DIR / "calibration_metric_summary.csv", index=False)
display(calibration_metric_summary)

# Score ablation: MSP, margin, and negative entropy are compared on the same calibration split.
score_ablation_rows = []
score_threshold_map = {
    "confidence_msp": THRESHOLD_GRID,
    "confidence_margin": np.round(np.arange(0.00, 1.001, 0.01), 2),
    "confidence_negative_entropy": np.round(np.linspace(cal_adv["confidence_negative_entropy"].min(), cal_adv["confidence_negative_entropy"].max(), 101), 6),
}
for score_col, grid in score_threshold_map.items():
    sweep = score_threshold_sweep(cal_adv, score_col, grid)
    sweep.to_csv(ADVANCED_THRESHOLD_DIR / f"threshold_sweep_{score_col}.csv", index=False)
    selected_score = select_from_sweep(sweep)
    selected_score["score"] = score_col
    score_ablation_rows.append(selected_score)
score_ablation = pd.DataFrame(score_ablation_rows)
score_ablation.to_csv(ADVANCED_THRESHOLD_DIR / "confidence_score_ablation.csv", index=False)
display(score_ablation)

# Primary risk-coverage and selected-threshold outputs.
primary_calibration_sweep = score_threshold_sweep(cal_adv, "phase1_confidence", THRESHOLD_GRID)
primary_calibration_sweep.to_csv(ADVANCED_THRESHOLD_DIR / "primary_threshold_sweep_temperature_scaled_msp.csv", index=False)

selected_fixed_test_metrics = selective_metrics_at_threshold(test_adv, "phase1_confidence", ROUTING_THRESHOLD)
selected_fixed_test_metrics["split"] = "test"
selected_fixed_calibration_metrics = selective_metrics_at_threshold(cal_adv, "phase1_confidence", ROUTING_THRESHOLD)
selected_fixed_calibration_metrics["split"] = "calibration"
fixed_threshold_summary = pd.DataFrame([selected_fixed_calibration_metrics, selected_fixed_test_metrics])
fixed_threshold_summary.to_csv(ADVANCED_THRESHOLD_DIR / "fixed_threshold_selective_metrics.csv", index=False)
display(fixed_threshold_summary)

risk_curve_cal, aurc_cal = risk_coverage_auc(cal_adv)
risk_curve_test, aurc_test = risk_coverage_auc(test_adv)
risk_curve_cal.to_csv(ADVANCED_THRESHOLD_DIR / "risk_coverage_curve_calibration_points.csv", index=False)
risk_curve_test.to_csv(ADVANCED_THRESHOLD_DIR / "risk_coverage_curve_test_points.csv", index=False)
pd.DataFrame([
    {"split": "calibration", "aurc": aurc_cal},
    {"split": "test", "aurc": aurc_test},
]).to_csv(ADVANCED_THRESHOLD_DIR / "aurc_summary.csv", index=False)

# Bootstrap CIs and threshold stability.
bootstrap_rows, bootstrap_summary = bootstrap_metrics(cal_adv, ROUTING_THRESHOLD, THRESHOLD_BOOTSTRAPS)
bootstrap_rows.to_csv(ADVANCED_THRESHOLD_DIR / "fixed_threshold_bootstrap_metrics.csv", index=False)
bootstrap_summary.to_csv(ADVANCED_THRESHOLD_DIR / "fixed_threshold_bootstrap_summary.csv", index=False)
threshold_stability = threshold_stability_bootstrap(cal_adv, THRESHOLD_BOOTSTRAPS)
threshold_stability.to_csv(ADVANCED_THRESHOLD_DIR / "threshold_bootstrap_stability.csv", index=False)
threshold_stability_summary = threshold_stability[["selected_threshold", "coverage", "selective_risk", "selective_risk_upper_bound"]].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "metric"})
threshold_stability_summary.to_csv(ADVANCED_THRESHOLD_DIR / "threshold_stability_summary.csv", index=False)

# High-confidence errors and per-class analysis.
high_confidence_errors = test_adv[(test_adv["phase1_confidence"] >= ROUTING_THRESHOLD) & (test_adv["label_str"] != test_adv["phase1_label"])].copy()
high_confidence_errors = high_confidence_errors.sort_values("phase1_confidence", ascending=False).head(TOP_HIGH_CONFIDENCE_ERRORS)
high_confidence_errors.to_csv(ADVANCED_THRESHOLD_DIR / "high_confidence_accepted_errors.csv", index=False)

per_class_metrics = per_class_selective_metrics(test_adv, ROUTING_THRESHOLD)
per_class_metrics.to_csv(ADVANCED_THRESHOLD_DIR / "per_class_selective_metrics_test.csv", index=False)
class_thresholds = class_conditional_thresholds(cal_adv)
class_thresholds.to_csv(ADVANCED_THRESHOLD_DIR / "class_conditional_thresholds.csv", index=False)

length_metrics = input_length_analysis(test_adv, ROUTING_THRESHOLD)
length_metrics.to_csv(ADVANCED_THRESHOLD_DIR / "input_length_threshold_analysis.csv", index=False)

# Paper-ready figures.
plt.figure(figsize=(5.5, 4.5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.plot(raw_ece_table["mean_confidence"], raw_ece_table["accuracy"], marker="o", label="Raw MSP")
plt.plot(scaled_ece_table["mean_confidence"], scaled_ece_table["accuracy"], marker="o", label="Temperature-scaled MSP")
plt.xlabel("Mean confidence")
plt.ylabel("Accuracy")
plt.title("Calibration Reliability Diagram")
plt.legend()
plt.tight_layout()
plt.savefig(ADVANCED_THRESHOLD_DIR / "calibration_reliability_diagram.png", dpi=200)
plt.show()

plt.figure(figsize=(5.8, 4.4))
plt.plot(primary_calibration_sweep["coverage"], primary_calibration_sweep["selective_risk"], label="Empirical risk")
plt.plot(primary_calibration_sweep["coverage"], primary_calibration_sweep["selective_risk_upper_bound"], label="One-sided upper bound")
plt.axhline(TARGET_SELECTIVE_RISK, color="red", linestyle="--", label=f"target risk={TARGET_SELECTIVE_RISK}")
plt.scatter([selected_fixed_calibration_metrics["coverage"]], [selected_fixed_calibration_metrics["selective_risk"]], color="black", zorder=5, label=f"selected tau={ROUTING_THRESHOLD:.2f}")
plt.xlabel("Coverage")
plt.ylabel("Selective risk")
plt.title("Calibration Risk-Coverage Curve")
plt.legend()
plt.tight_layout()
plt.savefig(ADVANCED_THRESHOLD_DIR / "risk_coverage_curve.png", dpi=200)
plt.show()

plt.figure(figsize=(5.8, 4.0))
sns.histplot(data=cal_adv, x="phase1_confidence", hue="label_str", bins=30, element="step")
plt.axvline(ROUTING_THRESHOLD, color="red", linestyle="--", label=f"threshold={ROUTING_THRESHOLD:.2f}")
plt.xlabel("Temperature-scaled MSP")
plt.title("Calibration Confidence Distribution")
plt.tight_layout()
plt.savefig(ADVANCED_THRESHOLD_DIR / "confidence_distribution_calibration.png", dpi=200)
plt.show()

# Reproducibility/provenance metadata.
threshold_provenance = {
    "model_name": MODEL_NAME,
    "samples_per_class": SAMPLES_PER_CLASS,
    "split_ratios": {
        "train": TRAIN_RATIO,
        "validation": VALIDATION_RATIO,
        "calibration": CALIBRATION_RATIO,
        "test": TEST_RATIO,
    },
    "confidence_method": "temperature_scaled_msp",
    "temperature": float(temperature),
    "selected_threshold": float(ROUTING_THRESHOLD),
    "target_selective_risk": float(TARGET_SELECTIVE_RISK),
    "risk_confidence_delta": float(RISK_CONFIDENCE_DELTA),
    "threshold_grid_min": float(np.min(THRESHOLD_GRID)),
    "threshold_grid_max": float(np.max(THRESHOLD_GRID)),
    "threshold_grid_step": 0.01,
    "selection_rule": "maximize calibration coverage subject to one-sided selective-risk upper bound <= target risk; fallback to lowest empirical risk then highest coverage",
    "advanced_output_dir": str(ADVANCED_THRESHOLD_DIR),
    "advanced_outputs": sorted([p.name for p in ADVANCED_THRESHOLD_DIR.glob("*")]),
}
(ADVANCED_THRESHOLD_DIR / "threshold_provenance.json").write_text(json.dumps(threshold_provenance, indent=2), encoding="utf-8")

advanced_summary = {
    **summary,
    "calibration_ece_raw_msp": raw_ece,
    "calibration_ece_temperature_scaled_msp": scaled_ece,
    "calibration_adaptive_ece_temperature_scaled_msp": adaptive_ece,
    "calibration_brier_temperature_scaled_msp": multiclass_brier_score(cal_probs_scaled, cal_y),
    "calibration_nll_temperature_scaled_msp": negative_log_likelihood(cal_probs_scaled, cal_y),
    "calibration_aurc": aurc_cal,
    "test_aurc": aurc_test,
    "advanced_threshold_output_dir": str(ADVANCED_THRESHOLD_DIR),
}
pd.DataFrame([advanced_summary]).to_csv(PHASE1_OUTPUT_DIR / "distilbert_phase1_advanced_summary.csv", index=False)
(PHASE1_OUTPUT_DIR / "distilbert_phase1_advanced_summary.json").write_text(json.dumps(advanced_summary, indent=2), encoding="utf-8")

print("Advanced confidence-threshold outputs saved to:")
print(ADVANCED_THRESHOLD_DIR)
print("Advanced output files:")
for output_file in sorted(ADVANCED_THRESHOLD_DIR.glob("*")):
    print("-", output_file.name)




## Mixed Emotion Phase 1 Predictions

In [ ]:
def load_mixed_emotion():
    if MIXED_EMOTION_LOCAL_PATH.exists():
        print("Loading Mixed Emotion from Drive:", MIXED_EMOTION_LOCAL_PATH)
        return pd.read_csv(MIXED_EMOTION_LOCAL_PATH)
    print("Loading Mixed Emotion from GitHub raw URL")
    return pd.read_csv(MIXED_EMOTION_DATA_URL)

mixed_df = load_mixed_emotion()
required = {"example_id", MIXED_TEXT_COLUMN, MIXED_LABEL_COLUMN}
missing = required - set(mixed_df.columns)
if missing:
    raise ValueError(f"Mixed Emotion data missing columns: {missing}")

mixed_for_model = pd.DataFrame({
    "example_id": mixed_df["example_id"],
    "text": mixed_df[MIXED_TEXT_COLUMN].astype(str),
    "label_str": mixed_df[MIXED_LABEL_COLUMN].astype(str),
})
mixed_for_model["label"] = mixed_for_model["label_str"].map(LABEL_TO_ID)
if mixed_for_model["label"].isna().any():
    raise ValueError("Mixed Emotion labels could not be mapped to the three-class label space.")
mixed_for_model["label"] = mixed_for_model["label"].astype(int)

mixed_logits, mixed_labels = predict_logits(mixed_for_model)
mixed_phase1 = prediction_frame(mixed_for_model, mixed_logits, temperature)
mixed_phase1["target_label"] = mixed_phase1["label_str"]
mixed_phase1["phase1_accepted"] = mixed_phase1["phase1_confidence"] >= ROUTING_THRESHOLD
mixed_phase1["phase1_routed"] = ~mixed_phase1["phase1_accepted"]
mixed_phase1["routing_threshold"] = ROUTING_THRESHOLD
mixed_phase1["temperature"] = temperature
# Preserve supplementary metadata for analysis.
metadata_cols = [c for c in mixed_df.columns if c not in mixed_phase1.columns and c not in {MIXED_TEXT_COLUMN, MIXED_LABEL_COLUMN}]
mixed_phase1 = mixed_phase1.merge(mixed_df[["example_id"] + metadata_cols], on="example_id", how="left")

mixed_phase1_path = PHASE1_OUTPUT_DIR / "phase1_mixed_emotion_predictions.csv"
mixed_phase1.to_csv(mixed_phase1_path, index=False)
print("Saved:", mixed_phase1_path)
print("Mixed Emotion Phase 1 accuracy:", accuracy_score(mixed_phase1["target_label"], mixed_phase1["phase1_label"]))
print("Routed count:", int(mixed_phase1["phase1_routed"].sum()), "/", len(mixed_phase1))
display(pd.crosstab(mixed_phase1["target_label"], mixed_phase1["phase1_label"], rownames=["target"], colnames=["phase1_label"]))
display(mixed_phase1.head())


## Paper-Ready Phase 1 Figures

In [ ]:
def save_confusion_matrix(df, pred_col, title, path):
    cm = confusion_matrix(df["target_label"], df[pred_col], labels=LABELS)
    plt.figure(figsize=(5.2, 4.2))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.xlabel("Predicted")
    plt.ylabel("Target")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.show()

save_confusion_matrix(mixed_phase1, "phase1_label", "Mixed Emotion Phase 1 DistilBERT", PHASE1_OUTPUT_DIR / "confusion_matrix_mixed_phase1_distilbert.png")

plt.figure(figsize=(6, 4))
sns.histplot(data=mixed_phase1, x="phase1_confidence", hue="target_label", bins=20, element="step")
plt.axvline(ROUTING_THRESHOLD, color="red", linestyle="--", label=f"threshold={ROUTING_THRESHOLD:.2f}")
plt.title("Mixed Emotion Phase 1 Confidence Distribution")
plt.tight_layout()
plt.savefig(PHASE1_OUTPUT_DIR / "mixed_phase1_confidence_distribution.png", dpi=200)
plt.show()


## Final Export

In [ ]:
OUTPUT_DIR_FOR_EXPORT = PHASE1_OUTPUT_DIR
EXPORT_ZIP_NAME = "distilbert_phase1_final_outputs"
# Final export / download cell.
# This creates one zip file containing all available outputs from this notebook.
from pathlib import Path
import zipfile

files_to_zip = []
for pattern in ["*.csv", "*.json", "*.png", "*.xlsx"]:
    files_to_zip.extend(sorted(OUTPUT_DIR_FOR_EXPORT.rglob(pattern)))

# Include saved model files if present, but avoid adding huge checkpoint internals repeatedly.
model_dir = globals().get("FINAL_MODEL_DIR")
if model_dir is not None and Path(model_dir).exists():
    for p in Path(model_dir).glob("*"):
        if p.is_file():
            files_to_zip.append(p)

print("Files found for export:")
for p in files_to_zip:
    print(f"- {p} | {p.stat().st_size:,} bytes")

if not files_to_zip:
    print("No output files found yet.")
else:
    zip_path = OUTPUT_DIR_FOR_EXPORT / f"{EXPORT_ZIP_NAME}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in files_to_zip:
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR_FOR_EXPORT) if p.is_relative_to(OUTPUT_DIR_FOR_EXPORT) else p.name)
    print(f"Saved zip: {zip_path} | {zip_path.stat().st_size:,} bytes")
    if DRIVE_OUTPUT_AVAILABLE:
        print("Persistent Drive copy is available here:")
        print(zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Automatic browser download was not started: {exc}")

